# Actor-Critic

Temporal-Difference Learning

V(s) = value-function - characterisitc of a state
q(s,a) = state-action function - characterisitc of an action given state

Actor = network that outputs policy
Critic = netowrk that predicts Value for the states



In [ ]:
def generate_sequence_with_actor(x, policy, max_len, eos_token, decode="sample"):
    """
    Roll out the actor to produce a sequence and log-probs.
    States are (x, prefix) pairs implicitly represented by the policy’s hidden state or context.
    """
    tokens = []
    logprobs = []
    states = []

    prefix = []  # generated so far
    state = policy.init_state(x)  # encoder context / KV cache / etc.

    for t in range(1, max_len + 1):
        # πθ(. | s_t)
        logits, state = policy.step(state, prefix)  # conditioned on x & prefix
        if decode == "sample":
            a_t, logp_t = sample_from_logits(logits)   # stochastic
        else:
            a_t, logp_t = argmax_from_logits(logits)   # greedy

        tokens.append(a_t)
        logprobs.append(logp_t)
        # Save a snapshot representing s_t (or pass the state object)
        states.append(snapshot_state(state, prefix))

        prefix.append(a_t)
        if a_t == eos_token:
            break

    return {"tokens": tokens, "logprobs": logprobs, "states": states}


In [ ]:
def compute_rewards(tokens_hat, y, score_fn, mixed_rollout=True, gamma=1.0):
    """
    Produce stepwise rewards r_t by differencing a sequence-level score over prefixes.
    We compute a prefix score S_t = score_fn(hybrid_prefix_t, y)
      - hybrid_prefix_t is either:
         (a) model prefix + ground-truth suffix (mixed rollout), OR
         (b) model prefix only (if mixed_rollout=False; usually noisier).
    Then r_t = S_t - S_{t-1}, with S_0 := score(empty, y) (often 0).
    """
    T_hat = len(tokens_hat)
    prefix_scores = []
    for t in range(1, T_hat + 1):
        model_prefix = tokens_hat[:t]

        if mixed_rollout:
            # append ground-truth suffix from t+1 .. |y|
            suffix = y[t:]  # 0-based indexing; ensure aligns with tokenization
            hybrid = model_prefix + suffix
        else:
            hybrid = model_prefix  # pure model prefix

        S_t = score_fn(hybrid, y)
        prefix_scores.append(S_t)

    S_prev = 0.0
    rewards = []
    for S_t in prefix_scores:
        r_t = S_t - S_prev
        rewards.append(r_t)
        S_prev = S_t

    return rewards


In [ ]:
def bootstrap_or_full_return(rewards, states, V, gamma=1.0, mode="MC"):
    """
    Build targets for the critic and effective returns for advantages.
    - MC:    targets[t] = R_t = sum_{k=t..T} γ^{k-t} r_k   (no bootstrapping)
    - TD(0): targets[t] = r_t + γ * V(s_{t+1})            (bootstrapped)
             with V(s_{T+1}) := 0
    Returns:
      targets: list of y_t^target for critic regression
      returns: list of R_t used to compute advantages A_t = R_t - V(s_t)
               (for TD mode we commonly reuse the bootstrapped target as the return)
    """
    T = len(rewards)
    targets = [0.0] * T
    returns = [0.0] * T

    if mode.upper() == "MC":
        G = 0.0
        # backward pass to accumulate full returns
        for t in reversed(range(T)):
            G = rewards[t] + gamma * G
            targets[t] = G   # critic fits full return
            returns[t] = G   # actor advantages use full return
        return targets, returns

    elif mode.upper() == "TD0":
        for t in range(T):
            r_t = rewards[t]
            if t == T - 1:
                boot = 0.0
            else:
                boot = V(states[t + 1])  # bootstrap from critic’s prediction
            y_target = r_t + gamma * boot
            targets[t] = y_target
            returns[t] = y_target  # common/simple choice
        return targets, returns

    else:
        raise ValueError("mode must be 'MC' or 'TD0'")


In [ ]:
def teacher_forced_cross_entropy(x, y, p_theta):
    """
    Standard SFT loss on (x, y):
      L_MLE = - (1/T) * sum_t log πθ(y_t | x, y_{<t})
    """
    state = p_theta.init_state(x)
    loss_terms = []
    prefix = []

    for t in range(len(y)):
        logits, state = p_theta.step(state, prefix)   # condition on ground-truth prefix
        logp = log_prob_of_token(logits, y[t])        # log πθ(y_t | x, y_{<t})
        loss_terms.append(-logp)
        prefix.append(y[t])

    return mean(loss_terms)


In [ ]:
def policy_sequence_entropies(states, p_theta):
    """
    Approximate per-step policy entropies H[πθ(.|s_t)] for an already generated rollout.
    """
    H = []
    for s in states:
        logits = p_theta.logits_from_state(s)
        probs = softmax(logits)
        H.append(-sum(probs * log(probs + 1e-12)))
    return H


In [ ]:
# =========================
# Notation / Assumptions
# =========================
# x: input/source sequence (e.g., src sentence)
# y: target/ground-truth sequence (tokens y[1..T])
# policy pθ: LM that defines πθ(a_t | s_t) and can:
#   - step() one token at a time
#   - compute log-probabilities of actions taken
# value Vφ: critic estimating V(s_t)
# score_fn(hat_y, y): sequence-level metric (e.g., BLEU in [0,1])
#   - For prefixes we’ll use "mixed rollouts": hat_y_prefix + ground_truth_suffix
# hyperparams: gamma, eta_actor, eta_critic, lambda_mle, beta_entropy, mode_returns
#   - mode_returns in {"MC", "TD0"} determines targets for critic (and A_t)

# =========================
# Core Training Loop
# =========================
for epoch in range(N_epochs):
    for (x, y) in training_data:

        # 1) Generate with current actor (sample or greedy)
        gen = generate_sequence_with_actor(x, policy=p_theta, max_len=MAX_LEN, eos_token=EOS)
        # gen = {
        #   "tokens": [a1, a2, ..., aT_hat],                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        
        #   "logprobs": [logπ(a1|s1), ..., logπ(aT_hat|sT_hat)],
        #   "states": [s1, s2, ..., sT_hat],
        # }

        # 2) Compute per-step rewards (intermediate + final) using mixed rollouts
        rewards = compute_rewards(gen["tokens"], y, score_fn, mixed_rollout=True)
        # rewards = [r1, r2, ..., rT_hat] ; often gamma=1.0 in sequence tasks

        # 3) Build return/target values (bootstrap or full returns)
        targets, returns = bootstrap_or_full_return(
            rewards=rewards,
            states=gen["states"],
            V=V_phi,
            gamma=gamma,
            mode=mode_returns  # "MC" (full returns) or "TD0" (bootstrapped)
        )
        # targets: critic regression targets at each t
        # returns: effective returns R_t (used for advantages)

        # 4) Critic update: fit Vφ(s_t) to targets
        V_pred = [V_phi(s) for s in gen["states"]]
        L_critic = mean([(V_pred[t] - targets[t])**2 for t in range(len(targets))])
        phi = phi - eta_critic * grad(L_critic, phi)

        # 5) Actor update: policy gradient with advantage baseline
        advantages = [returns[t] - stop_grad(V_pred[t]) for t in range(len(returns))]
        # Policy loss (REINFORCE with baseline)
        L_pg = - mean([advantages[t] * gen["logprobs"][t] for t in range(len(gen["logprobs"]))])

        # Entropy bonus (to encourage exploration)
        entropies = policy_sequence_entropies(gen["states"], p_theta)  # per-step H[πθ(.|s_t)]
        L_entropy = - beta_entropy * mean(entropies)

        # 6) Mix with MLE loss (teacher-forcing cross-entropy on (x, y))
        #    Compute over the same (x, y) pair using standard SFT objective.
        L_mle = teacher_forced_cross_entropy(x, y, p_theta)

        # Final actor loss
        L_actor = L_pg + L_entropy + lambda_mle * L_mle
        theta = theta - eta_actor * grad(L_actor, theta)

        # (Optional) Soft-update a target critic network for stability
        # V_phi_target ← τ * V_phi + (1-τ) * V_phi_target
